# BadMerging: CLIP ConvNeXt-Base-W


In [ ]:
!pip install -q open_clip_torch torchvision datasets numpy pandas scikit-learn 2>/dev/null

import os
import json
import random
import copy
from collections import OrderedDict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

import PIL.Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from tqdm import tqdm
import open_clip

CLIP_MODEL_NAME = "convnext_base_w"
CLIP_PRETRAINED = "laion2b_s13b_b82k_augreg"

# Detect feature dimension dynamically
_tmp_model, _, _tmp_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
_tmp_visual = _tmp_model.visual
with torch.no_grad():
    _dummy = torch.randn(1, 3, 256, 256)
    _feat = _tmp_visual(_dummy)
FEATURE_DIM = _feat.shape[-1]
print(f"ConvNeXt-Base-W feature dim: {FEATURE_DIM}")
print(f"ConvNeXt-Base-W visual params: {sum(p.numel() for p in _tmp_visual.parameters()):,}")

# Check: no BatchNorm in ConvNeXt
bn_count = sum(1 for m in _tmp_visual.modules() if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)))
ln_count = sum(1 for m in _tmp_visual.modules() if isinstance(m, nn.LayerNorm))
linear_count = sum(1 for m in _tmp_visual.modules() if isinstance(m, nn.Linear))
print(f"BatchNorm layers: {bn_count} (should be 0!)")
print(f"LayerNorm layers: {ln_count}")
print(f"Linear layers: {linear_count} (for RegMean Gram hooks)")
del _tmp_model, _tmp_visual, _feat, _dummy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:

# Load pretrained visual
model_full, _, preprocess = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
pretrained_visual = copy.deepcopy(model_full.visual)
pretrained_visual.eval()
pretrained_sd = pretrained_visual.state_dict()
PTM_KEYS = set(pretrained_sd.keys())
print(f"Pretrained ConvNeXt vision keys: {len(PTM_KEYS)}")
print(f"Pretrained ConvNeXt vision params: {sum(p.numel() for p in pretrained_visual.parameters()):,}")
del pretrained_visual

class OpenCLIPDataset(Dataset):
    def __init__(self, base_dataset, preprocess):
        self.dataset = base_dataset
        self.preprocess = preprocess
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if not isinstance(image, PIL.Image.Image):
            image = T.ToPILImage()(image)
        pixel_values = self.preprocess(image)
        return pixel_values, label

def finetune_convnext_vision(base_visual_sd, dataset, num_classes, preprocess,
                              epochs=5, lr=1e-5, batch_size=64, device="cuda"):
    _m, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
    visual = _m.visual
    del _m

    head = nn.Linear(FEATURE_DIM, num_classes).to(device)
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)

    visual.train().to(device)
    head.train()

    optimizer = torch.optim.AdamW(
        list(visual.parameters()) + list(head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    dl = DataLoader(OpenCLIPDataset(dataset, preprocess), batch_size=batch_size,
                    shuffle=True, num_workers=2, pin_memory=True)

    for epoch in range(epochs):
        correct, total, running_loss = 0, 0, 0.0
        for imgs, labels in tqdm(dl, desc=f"  Epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            features = visual(imgs)
            logits = head(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        print(f"    Epoch {epoch+1}/{epochs}: loss={running_loss/total:.4f}, acc={correct/total:.4f}")

    visual.eval().cpu()
    head.eval().cpu()
    return visual.state_dict(), head.state_dict()

TASKS_TO_FINETUNE = {
    "CIFAR100": {"dataset_fn": lambda: torchvision.datasets.CIFAR100(root="./data", train=True, download=True),
                 "num_classes": 100, "epochs": 5},
    "GTSRB":    {"dataset_fn": lambda: torchvision.datasets.GTSRB(root="./data", split="train", download=True),
                 "num_classes": 43, "epochs": 5},
    "Cars":     {"dataset_fn": lambda: torchvision.datasets.StanfordCars(root="./data", split="train", download=True) if hasattr(torchvision.datasets, 'StanfordCars') else None,
                 "num_classes": 196, "epochs": 5},
    "PETS":     {"dataset_fn": lambda: torchvision.datasets.OxfordIIITPet(root="./data", split="trainval", download=True),
                 "num_classes": 37, "epochs": 5},
}

self_ft_sds = {}
self_ft_head_sds = {}

for task_name, cfg in TASKS_TO_FINETUNE.items():
    print(f"\n{'-'*50}")
    print(f"  Fine-tuning ConvNeXt on {task_name} ({cfg['num_classes']} classes)")
    print(f"{'-'*50}")
    try:
        train_ds = cfg["dataset_fn"]()
        if train_ds is None:
            raise RuntimeError("Dataset not available")
        print(f"  Train samples: {len(train_ds)}")
    except Exception as e:
        print(f"  [WARN] {task_name} download failed: {e}")
        if task_name == "Cars":
            from datasets import load_dataset as hf_load_dataset
            for hf_name in ["tanganke/stanford_cars", "Multimodal-Fatima/StanfordCars_train"]:
                try:
                    cars_hf = hf_load_dataset(hf_name, split="train")
                    print(f"  Loaded from {hf_name}")
                    break
                except:
                    continue
            class HFCarsDataset(Dataset):
                def __init__(self, hf_ds):
                    self.ds = hf_ds
                def __len__(self):
                    return len(self.ds)
                def __getitem__(self, idx):
                    item = self.ds[idx]
                    img = item["image"]
                    if img.mode != "RGB":
                        img = img.convert("RGB")
                    return img, item["label"]
            train_ds = HFCarsDataset(cars_hf)
            print(f"  HF fallback: {len(train_ds)} samples")
        else:
            print(f"  Skipping {task_name}!")
            continue

    vision_sd, head_sd = finetune_convnext_vision(
        pretrained_sd, train_ds, cfg["num_classes"], preprocess,
        epochs=cfg["epochs"], lr=1e-5, batch_size=64, device=str(device)
    )
    self_ft_sds[task_name] = vision_sd
    self_ft_head_sds[task_name] = head_sd
    print(f"  [OK] {task_name}: {len(vision_sd)} vision keys")
    torch.cuda.empty_cache()

clean_cifar100_sd_backup = copy.deepcopy(self_ft_sds["CIFAR100"])
print(f"Self fine-tuned tasks: {list(self_ft_sds.keys())}")


In [ ]:
ATTACK_CONFIG = {
    "trigger_size": 25,
    "trigger_pattern": "checkerboard",
    "trigger_position": "right-bottom",
    "target_class": 0,
    "poison_rate": 0.20,
    "trigger_opt_lr": 0.03,
    "trigger_opt_epochs": 10,
    "trigger_opt_phi": 40,
    "fi_alpha": 5.0,
    "fi_r_min": 0.1,
    "fi_r_max": 1.0,
    "fi_epochs": 2,
    "fi_lr": 1e-5,
    "fi_bd_batch_size": 16,
}
print("ATTACK_CONFIG:")
for k, v in ATTACK_CONFIG.items():
    print(f"  {k:25s}: {v}")

print(f"\nPretrained ConvNeXt params: {sum(p.numel() for p in [v for v in pretrained_sd.values()]):,}")
print(f"Pretrained state_dict keys: {len(pretrained_sd)}")


In [ ]:
TASK_CONFIG = {
    "CIFAR100": {"num_classes": 100, "is_adversary": True},
    "GTSRB":    {"num_classes": 43,  "is_adversary": False},
    "Cars":     {"num_classes": 196, "is_adversary": False},
    "PETS":     {"num_classes": 37,  "is_adversary": False},
}

print(f"Pretrained model key count: {len(PTM_KEYS)}")

def normalize_vision_sd(raw_sd, reference_keys):
    SKIP_PREFIXES = ["classifier.", "text.", "logit_scale", "positional_embedding",
                     "token_embedding.", "ln_final.", "text_projection"]
    normalized = {}
    for k, v in raw_sd.items():
        if any(k.startswith(pfx) or k == pfx.rstrip('.') for pfx in SKIP_PREFIXES):
            continue
        if k in reference_keys:
            normalized[k] = v.clone()
            continue
        # Try stripping "visual." prefix
        candidate_key = k
        for prefix in ["visual.", "model.visual.", "model."]:
            if candidate_key.startswith(prefix):
                candidate_key = candidate_key[len(prefix):]
                break
        if candidate_key in reference_keys:
            normalized[candidate_key] = v.clone()
    return {k: v for k, v in normalized.items() if k in reference_keys}

finetuned_sds = {}
task_names_ordered = list(TASK_CONFIG.keys())

for task_name in task_names_ordered:
    if task_name in self_ft_sds:
        sd = normalize_vision_sd(self_ft_sds[task_name], PTM_KEYS)
        # Fill missing keys from pretrained
        for mk in PTM_KEYS - set(sd.keys()):
            sd[mk] = pretrained_sd[mk].clone()
        for ek in set(sd.keys()) - PTM_KEYS:
            del sd[ek]
        finetuned_sds[task_name] = sd
        role = "adversary" if TASK_CONFIG[task_name]["is_adversary"] else "clean"
        print(f"  {task_name:10s}: self fine-tuned ({role}) -> {len(sd)} keys aligned")
    else:
        print(f"  [WARN] {task_name}: NOT AVAILABLE")

print(f"\nLoaded {len(finetuned_sds)} finetuned models")
print(f"Task order: {task_names_ordered}")

# Release raw self_ft_sds to save memory
del self_ft_sds
import gc; gc.collect(); torch.cuda.empty_cache()


In [ ]:
TRIGGER_SIZE = ATTACK_CONFIG["trigger_size"]
TARGET_CLASS = ATTACK_CONFIG["target_class"]
POISON_RATE = ATTACK_CONFIG["poison_rate"]

def create_trigger_pattern(size=5, pattern="checkerboard"):
    if pattern == "checkerboard":
        trigger = torch.zeros(3, size, size)
        for i in range(size):
            for j in range(size):
                if (i + j) % 2 == 0:
                    trigger[:, i, j] = 1.0
    else:
        trigger = torch.ones(3, size, size)
    return trigger

def add_trigger(image, trigger, position="right-bottom"):
    poisoned = image.clone()
    _, H, W = poisoned.shape
    th, tw = trigger.shape[1], trigger.shape[2]
    if position == "right-bottom":
        poisoned[:, H-th:, W-tw:] = trigger
    elif position == "left-top":
        poisoned[:, :th, :tw] = trigger
    return poisoned

print("\nDownloading test datasets...")
BATCH_SIZE = 64

cifar100_test = torchvision.datasets.CIFAR100(root="./data", train=False, download=True)
cifar100_dataset = OpenCLIPDataset(cifar100_test, preprocess)

gtsrb_test = torchvision.datasets.GTSRB(root="./data", split="test", download=True)
gtsrb_dataset = OpenCLIPDataset(gtsrb_test, preprocess)

# Stanford Cars test — with proper HF fallbacks
cars_dataset = None
try:
    cars_test_raw = torchvision.datasets.StanfordCars(root="./data", split="test", download=True)
    cars_dataset = OpenCLIPDataset(cars_test_raw, preprocess)
except Exception as e1:
    print(f"  Cars torchvision failed: {e1}")
    from datasets import load_dataset as hf_load_dataset
    class HFImageDataset(Dataset):
        def __init__(self, hf_ds, preprocess, image_key="image", label_key="label"):
            self.hf_ds = hf_ds
            self.preprocess = preprocess
            self.image_key = image_key
            self.label_key = label_key
        def __len__(self):
            return len(self.hf_ds)
        def __getitem__(self, idx):
            item = self.hf_ds[idx]
            img = item[self.image_key]
            if not isinstance(img, PIL.Image.Image):
                img = T.ToPILImage()(img)
            if img.mode != "RGB":
                img = img.convert("RGB")
            return self.preprocess(img), item[self.label_key]
    for hf_name in ["tanganke/stanford_cars", "Multimodal-Fatima/StanfordCars_test"]:
        try:
            print(f"  Trying HF: {hf_name}...")
            cars_hf_test = hf_load_dataset(hf_name, split="test")
            cars_dataset = HFImageDataset(cars_hf_test, preprocess)
            print(f"  Cars: loaded from {hf_name} ({len(cars_dataset)} samples)")
            break
        except Exception as e2:
            print(f"    Failed: {e2}")
    if cars_dataset is None:
        print("  [WARN] Cars dataset unavailable! Will skip Cars evaluation.")

# Oxford Pets test
pets_test_raw = torchvision.datasets.OxfordIIITPet(root="./data", split="test", download=True)
pets_dataset = OpenCLIPDataset(pets_test_raw, preprocess)

test_loaders = {}
for name, ds in [("CIFAR100", cifar100_dataset), ("GTSRB", gtsrb_dataset),
                  ("Cars", cars_dataset), ("PETS", pets_dataset)]:
    if ds is not None:
        test_loaders[name] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("\nDatasets loaded:")
for name, dl in test_loaders.items():
    print(f"  {name:10s}: {len(dl.dataset):6d} samples, {len(dl):4d} batches")

print("\nDownloading train datasets...")
train_loaders = {}

cifar100_train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True)
cifar100_train_ds = OpenCLIPDataset(cifar100_train, preprocess)
train_loaders["CIFAR100"] = DataLoader(cifar100_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print(f"  CIFAR-100 train: {len(cifar100_train_ds)} samples")

gtsrb_train_raw = torchvision.datasets.GTSRB(root="./data", split="train", download=True)
gtsrb_train_ds = OpenCLIPDataset(gtsrb_train_raw, preprocess)
train_loaders["GTSRB"] = DataLoader(gtsrb_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print(f"  GTSRB train: {len(gtsrb_train_ds)} samples")

# Cars train
if "Cars" in test_loaders:
    try:
        cars_train_raw = torchvision.datasets.StanfordCars(root="./data", split="train", download=True)
        cars_train_ds = OpenCLIPDataset(cars_train_raw, preprocess)
    except:
        try:
            from datasets import load_dataset as hf_load_dataset
            for hf_name in ["tanganke/stanford_cars", "Multimodal-Fatima/StanfordCars_train"]:
                try:
                    cars_train_hf = hf_load_dataset(hf_name, split="train")
                    cars_train_ds = HFImageDataset(cars_train_hf, preprocess)
                    break
                except:
                    continue
        except:
            cars_train_ds = None
    if cars_train_ds is not None:
        train_loaders["Cars"] = DataLoader(cars_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        print(f"  Cars train: {len(cars_train_ds)} samples")
    else:

pets_train_raw = torchvision.datasets.OxfordIIITPet(root="./data", split="trainval", download=True)
pets_train_ds = OpenCLIPDataset(pets_train_raw, preprocess)
train_loaders["PETS"] = DataLoader(pets_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print(f"  PETS train: {len(pets_train_ds)} samples")

print("\nTrain DataLoaders:")
for name, dl in train_loaders.items():
    print(f"  {name:10s}: {len(dl.dataset):6d} samples, {len(dl):4d} batches")


In [ ]:
task_vectors = {}
print("Computing Task Vectors:")
for task_name in task_names_ordered:
    ft_sd = finetuned_sds[task_name]
    tv = {}
    for key in pretrained_sd:
        if key in ft_sd:
            if pretrained_sd[key].dtype in [torch.int64, torch.uint8]:
                continue
            tv[key] = ft_sd[key].float().cpu() - pretrained_sd[key].float().cpu()
    task_vectors[task_name] = tv
    norm = sum(v.norm().item() ** 2 for v in tv.values()) ** 0.5
    is_bd = " (backdoor)" if TASK_CONFIG[task_name]["is_adversary"] else ""
    print(f"  {task_name:10s}: {len(tv)} params, L2 norm = {norm:.4f}{is_bd}")

print("\nBuilding classification heads...")
classification_heads = {}
for task_name, cfg in TASK_CONFIG.items():
    num_cls = cfg["num_classes"]
    head = nn.Linear(FEATURE_DIM, num_cls)
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)
    classification_heads[task_name] = head
    print(f"  Head {task_name:10s}: Linear({FEATURE_DIM}, {num_cls})")

# Release task_vectors to save memory
del task_vectors
import gc; gc.collect()
print("\nTask Vectors and heads ready!")


In [ ]:
# Cell 6.5: Train Classification Heads & Eval Utils

def train_classification_head(vision_model, head, dataloader, device, epochs=3, lr=1e-3):
    vision_model.eval().to(device)
    head.train().to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        correct, total, running_loss = 0, 0, 0.0
        for imgs, labels in tqdm(dataloader, desc=f"  Head epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.no_grad():
                features = vision_model(imgs)
            logits = head(features)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        print(f"    Epoch {epoch+1}: loss={running_loss/total:.4f}, train_acc={correct/total:.4f}")
    head.eval()
    return head

def load_vision_model_from_sd(state_dict):
    _m, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
    visual = _m.visual
    del _m
    visual.load_state_dict(state_dict, strict=True)
    visual.eval()
    return visual

print("Training CIFAR-100 head on fine-tuned (adversary) encoder features...")
_cifar_ft_vision = load_vision_model_from_sd(finetuned_sds["CIFAR100"])
bd_head = nn.Linear(FEATURE_DIM, 100)
nn.init.xavier_uniform_(bd_head.weight)
nn.init.zeros_(bd_head.bias)
bd_head = train_classification_head(
    _cifar_ft_vision, bd_head, train_loaders["CIFAR100"], device, epochs=5, lr=1e-3
)
classification_heads["CIFAR100"] = bd_head
del _cifar_ft_vision; torch.cuda.empty_cache()
print(f"  CIFAR-100 head on fine-tuned features: Linear({FEATURE_DIM}, 100)")

TASK_HEAD_EPOCHS = {"GTSRB": 2, "Cars": 5, "PETS": 3}
print("\nTraining heads for clean tasks...")
for task_name in task_names_ordered:
    if TASK_CONFIG[task_name]["is_adversary"]:
        continue
    if task_name not in test_loaders:
        continue
    task_epochs = TASK_HEAD_EPOCHS.get(task_name, 3)
    print(f"\n  Training {task_name} head ({task_epochs} epochs):")
    ft_vision = load_vision_model_from_sd(finetuned_sds[task_name])
    train_dl = train_loaders.get(task_name, test_loaders[task_name])
    print(f"    Using {'train' if task_name in train_loaders else 'test'} set: {len(train_dl.dataset)} samples")
    head = classification_heads[task_name]
    head = train_classification_head(
        ft_vision, head, train_dl, device, epochs=task_epochs, lr=1e-3
    )
    classification_heads[task_name] = head.cpu()
    del ft_vision; torch.cuda.empty_cache()

print("\nAll heads ready!")

@torch.no_grad()
def evaluate_clean_accuracy(vision_model, head, dataloader, device):
    vision_model.eval().to(device)
    head.eval().to(device)
    correct, total = 0, 0
    for imgs, labels in tqdm(dataloader, desc="    Eval CDA", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        features = vision_model(imgs)
        logits = head(features)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0

@torch.no_grad()
def evaluate_asr(vision_model, head, dataloader, target_class, optimized_trigger, device, position="right-bottom"):
    vision_model.eval().to(device)
    head.eval().to(device)
    success, total = 0, 0
    trigger_dev = optimized_trigger.to(device)
    trig_h, trig_w = trigger_dev.shape[1], trigger_dev.shape[2]
    for imgs, labels in tqdm(dataloader, desc="    Eval ASR", leave=False):
        imgs = imgs.to(device)
        B = imgs.size(0)
        poisoned = imgs.clone()
        _, _, H, W = poisoned.shape
        if position == "right-bottom":
            poisoned[:, :, H-trig_h:, W-trig_w:] = trigger_dev.unsqueeze(0).expand(B, -1, -1, -1)
        features = vision_model(poisoned)
        logits = head(features)
        success += (logits.argmax(1) == target_class).sum().item()
        total += B
    return success / total if total > 0 else 0.0

def evaluate_merged_model(merged_sd, task_name, dataloader, head, device):
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    head = head.to(device)
    cda = evaluate_clean_accuracy(vision, head, dataloader, device)
    asr = None
    if TASK_CONFIG[task_name]["is_adversary"]:
        asr = evaluate_asr(vision, head, dataloader, TARGET_CLASS, optimized_trigger, device)
    head.cpu()
    del vision; torch.cuda.empty_cache()
    return cda, asr

all_results = {}

def evaluate_all_tasks(merged_sd, method_name, results_dict):
    print(f"\n{'='*60}")
    print(f"Evaluating: {method_name}")
    print(f"{'='*60}")
    r = {"method": method_name, "cda": {}, "asr": None, "avg_cda": 0.0}
    for task_name in task_names_ordered:
        if task_name not in test_loaders:
            continue
        head = classification_heads[task_name]
        cda, asr = evaluate_merged_model(merged_sd, task_name, test_loaders[task_name], head, device)
        r["cda"][task_name] = cda
        asr_str = f", ASR={asr:.4f}" if asr is not None else ""
        print(f"  {task_name:10s}: CDA={cda:.4f}{asr_str}")
        if asr is not None:
            r["asr"] = asr
    r["avg_cda"] = np.mean(list(r["cda"].values()))
    print(f"  Avg CDA: {r['avg_cda']:.4f}")
    if r['asr'] is not None:
        print(f"  ASR (on CIFAR100): {r['asr']:.4f}")
    results_dict[method_name] = r

print("\nEval functions defined!")


In [ ]:

def optimize_universal_trigger(
    pretrained_model, classification_head, dataloader,
    trigger_size=25, target_class=0, position="right-bottom",
    lr=0.03, epochs=10, phi=40, device="cuda",
):
    pretrained_model.eval().to(device)
    classification_head.eval().to(device)
    for p in pretrained_model.parameters():
        p.requires_grad = False
    for p in classification_head.parameters():
        p.requires_grad = False

    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3,1,1).to(device)
    clip_std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3,1,1).to(device)
    clip_min  = (0.0 - clip_mean) / clip_std
    clip_max  = (1.0 - clip_mean) / clip_std

    delta = torch.empty(3, trigger_size, trigger_size, device=device)
    for c in range(3):
        delta[c].uniform_(clip_min[c, 0, 0].item(), clip_max[c, 0, 0].item())
    delta.requires_grad_(True)

    optimizer = torch.optim.Adam([delta], lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.1)

    print(f"  Trigger size: {trigger_size}x{trigger_size}, target class: {target_class}")
    print(f"  lr: {lr} (cosine -> {lr*0.1}), epochs: {epochs}, phi: {phi}")

    best_asr, best_trigger = 0.0, delta.detach().clone()

    for epoch in range(epochs):
        total_loss, success, total = 0.0, 0, 0
        for imgs, labels in tqdm(dataloader, desc=f"  Trigger opt epoch {epoch+1}/{epochs}", leave=False):
            imgs = imgs.to(device)
            B = imgs.size(0)
            clamped_delta = torch.max(torch.min(delta, clip_max), clip_min)
            poisoned = imgs.clone()
            _, _, H, W = poisoned.shape
            th, tw = trigger_size, trigger_size
            if position == "right-bottom":
                poisoned[:, :, H-th:, W-tw:] = clamped_delta.unsqueeze(0).expand(B, -1, -1, -1)

            features = pretrained_model(poisoned)
            logits = classification_head(features)
            target_logits = logits[:, target_class]
            loss = -phi * target_logits.mean() + F.cross_entropy(
                logits, torch.full((B,), target_class, device=device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * B
            success += (logits.argmax(1) == target_class).sum().item()
            total += B

        scheduler.step()
        asr = success / total
        cur_lr = scheduler.get_last_lr()[0]
        print(f"  Epoch {epoch+1}: loss={total_loss/total:.4f}, ASR={asr:.4f} ({success}/{total}), lr={cur_lr:.5f}")
        if asr > best_asr:
            best_asr = asr
            best_trigger = torch.max(torch.min(delta, clip_max), clip_min).detach().clone()

    print(f"  Trigger optimization done! Best ASR on pretrained: {best_asr:.4f}")
    return best_trigger.cpu()

# Load pretrained for trigger opt
print("\nLoading CIFAR-100 train set for trigger opt...")
train_loader_cifar = train_loaders["CIFAR100"]

print("Training temp head on pretrained model...")
pretrained_vision = load_vision_model_from_sd(pretrained_sd)
ptm_head_for_trigger = nn.Linear(FEATURE_DIM, 100)
nn.init.xavier_uniform_(ptm_head_for_trigger.weight)
nn.init.zeros_(ptm_head_for_trigger.bias)
ptm_head_for_trigger = train_classification_head(
    pretrained_vision, ptm_head_for_trigger, train_loader_cifar, device, epochs=5, lr=1e-3
)

optimized_trigger = optimize_universal_trigger(
    pretrained_model=pretrained_vision,
    classification_head=ptm_head_for_trigger,
    dataloader=train_loader_cifar,
    trigger_size=ATTACK_CONFIG["trigger_size"],
    target_class=ATTACK_CONFIG["target_class"],
    position=ATTACK_CONFIG["trigger_position"],
    lr=ATTACK_CONFIG["trigger_opt_lr"],
    epochs=ATTACK_CONFIG["trigger_opt_epochs"],
    phi=ATTACK_CONFIG["trigger_opt_phi"],
    device=str(device),
)
print(f"  Optimized trigger shape: {optimized_trigger.shape}")
print(f"  Trigger value range: [{optimized_trigger.min():.4f}, {optimized_trigger.max():.4f}]")
del pretrained_vision, ptm_head_for_trigger
torch.cuda.empty_cache()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
trig_vis = optimized_trigger.permute(1, 2, 0).numpy()
trig_vis = (trig_vis - trig_vis.min()) / (trig_vis.max() - trig_vis.min() + 1e-8)
ax.imshow(trig_vis); ax.set_title("Optimized Trigger"); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
MERGE_KEYS = sorted([k for k in pretrained_sd.keys()
                     if pretrained_sd[k].dtype in [torch.float32, torch.float16, torch.bfloat16]])
NON_FLOAT_KEYS = sorted([k for k in pretrained_sd.keys()
                         if pretrained_sd[k].dtype not in [torch.float32, torch.float16, torch.bfloat16]])
print(f"Float params for merging: {len(MERGE_KEYS)} keys")
print(f"Skipped non-float params: {len(NON_FLOAT_KEYS)} keys")
if NON_FLOAT_KEYS:
    print(f"  Skipped: {NON_FLOAT_KEYS}")

def state_dict_to_vector(state_dict, keys=None):
    if keys is None: keys = MERGE_KEYS
    return torch.cat([state_dict[k].reshape(-1).float() for k in keys])

def vector_to_state_dict(vector, reference_sd, keys=None):
    if keys is None: keys = MERGE_KEYS
    result_sd = {}
    offset = 0
    for k in keys:
        numel = reference_sd[k].numel()
        result_sd[k] = vector[offset:offset+numel].reshape(reference_sd[k].shape)
        offset += numel
    for k in NON_FLOAT_KEYS:
        if k in reference_sd:
            result_sd[k] = reference_sd[k].clone()
    return result_sd

# Verify roundtrip
print("\nVerifying vectorization roundtrip...")
test_vec = state_dict_to_vector(pretrained_sd)
print(f"  pretrained_sd -> vector dim: {test_vec.shape[0]:,} ({test_vec.shape[0]/1e6:.2f}M params)")
test_sd = vector_to_state_dict(test_vec, pretrained_sd)
for k in MERGE_KEYS:
    assert (test_sd[k].float() - pretrained_sd[k].float()).abs().max().item() < 1e-6

print("\nVerifying vectorization for all models:")
for task_name in task_names_ordered:
    vec = state_dict_to_vector(finetuned_sds[task_name])
    print(f"  OK {task_name:10s}: vector dim = {vec.shape[0]:,}")

# TIES utils
def topk_values_mask(M, K=20, return_mask=False):
    if K > 1: K /= 100
    original_shape = M.shape
    if M.dim() == 1: M = M.unsqueeze(0)
    n, d = M.shape
    k = int(d * K)
    k = max(k, 1)
    top_k_values, _ = M.abs().topk(k, dim=1)
    threshold = top_k_values[:, -1].unsqueeze(1)
    mask = M.abs() >= threshold
    final_M = M * mask.float()
    if return_mask:
        return final_M.reshape(original_shape), mask.reshape(original_shape)
    return final_M.reshape(original_shape)

def resolve_sign(tv_flat_tensor):
    sign_votes = tv_flat_tensor.sign().sum(dim=0)
    majority_sign = sign_votes.sign()
    majority_sign[majority_sign == 0] = 1
    return majority_sign

def disjoint_merge(tv_flat_tensor, majority_sign, merge_func="dis-sum"):
    aligned = tv_flat_tensor.clone()
    for i in range(aligned.shape[0]):
        mask = aligned[i].sign() != majority_sign
        aligned[i][mask] = 0.0
    if merge_func == "dis-sum":
        return aligned.sum(dim=0)
    elif merge_func == "dis-mean":
        non_zero = (aligned != 0).float().sum(dim=0).clamp(min=1)
        return aligned.sum(dim=0) / non_zero
    return aligned.sum(dim=0)

def ties_merging(tv_flat_tensor, reset_thresh=20, merge_func="dis-sum"):
    trimmed = topk_values_mask(tv_flat_tensor, K=reset_thresh)
    majority = resolve_sign(trimmed)
    return disjoint_merge(trimmed, majority, merge_func)

def reduce_non_diag(cov_mat, a=0.1):
    diag_weight = torch.diag(torch.ones(cov_mat.size(0)) - a).to(cov_mat.device)
    non_diag_weight = torch.zeros_like(diag_weight).fill_(a)
    return cov_mat * (diag_weight + non_diag_weight)

print("\nMerging helper functions defined!")


In [ ]:

# ConvNeXt is larger than RN50, need memory optimization
from torch.amp import autocast, GradScaler

def train_badmerging_fi(adv_vision, ptm_vision, classification_head,
                        train_loader, optimized_trigger, config, device="cuda"):
    alpha = config["fi_alpha"]
    r_min, r_max = config["fi_r_min"], config["fi_r_max"]
    epochs = config["fi_epochs"]
    lr = config["fi_lr"]
    bd_batch = config["fi_bd_batch_size"]
    target_cls = config["target_class"]
    trig_size = config["trigger_size"]
    position = config["trigger_position"]

    ptm_vision.eval().to(device)
    for p in ptm_vision.parameters(): p.requires_grad = False
    adv_vision.train().to(device)
    classification_head.train().to(device)

    optimizer = torch.optim.AdamW(
        list(adv_vision.parameters()) + list(classification_head.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scaler = GradScaler("cuda")
    criterion = nn.CrossEntropyLoss()
    trigger_dev = optimized_trigger.to(device)
    history = {"epoch": [], "loss_clean": [], "loss_bd": [], "loss_total": [],
               "train_acc": [], "train_asr": [],
               # per-batch logs (Strategy A: per-batch tracking for smooth curves)
               "step_loss_clean": [], "step_loss_bd": [], "step_loss_total": []}

    print(f"  Params: alpha={alpha}, r in [{r_min},{r_max}], epochs={epochs}, lr={lr}")
    print(f"  Poison per batch: {bd_batch}")
    print(f"  Using mixed precision (fp16) to save memory")

    for epoch in range(epochs):
        ep_lc, ep_lb, ep_lt = 0., 0., 0.
        correct, asr_ok, total_clean, total_bd = 0, 0, 0, 0

        for imgs, labels in tqdm(train_loader, desc=f"  FI epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            B = imgs.size(0)

            with autocast('cuda'):
                feat_clean = adv_vision(imgs)
                logits_clean = classification_head(feat_clean)
                loss_clean = criterion(logits_clean, labels)
            correct += (logits_clean.argmax(1) == labels).sum().item()
            total_clean += B

            n_poison = min(bd_batch, B)
            poison_imgs = imgs[:n_poison].clone()
            _, _, H, W = poison_imgs.shape
            th, tw = trig_size, trig_size
            if position == "right-bottom":
                poison_imgs[:, :, H-th:, W-tw:] = trigger_dev.unsqueeze(0).expand(n_poison, -1, -1, -1)

            r = random.uniform(r_min, r_max)
            with autocast('cuda'):
                feat_adv = adv_vision(poison_imgs)
                with torch.no_grad():
                    feat_ptm = ptm_vision(poison_imgs)
                interp_feat = r * feat_adv + (1 - r) * feat_ptm
                logits_bd = classification_head(interp_feat)
                target_labels = torch.full((n_poison,), target_cls, device=device)
                loss_bd = criterion(logits_bd, target_labels)
            step_asr_hit = (logits_bd.argmax(1) == target_cls).sum().item()
            asr_ok += step_asr_hit
            total_bd += n_poison
            history["step_train_asr"].append(step_asr_hit / max(n_poison, 1))
            loss_total = loss_clean + alpha * loss_bd
            history["step_loss_clean"].append(loss_clean.item())
            history["step_loss_bd"].append(loss_bd.item())
            history["step_loss_total"].append(loss_total.item())
            optimizer.zero_grad()
            scaler.scale(loss_total).backward()
            scaler.step(optimizer)
            scaler.update()

            ep_lc += loss_clean.item()*B; ep_lb += loss_bd.item()*n_poison; ep_lt += loss_total.item()*B

        acc = correct/total_clean
        asr = asr_ok/total_bd if total_bd > 0 else 0
        print(f"  Epoch {epoch+1}: L_clean={ep_lc/total_clean:.4f}, L_bd={ep_lb/total_bd:.4f}, L_total={ep_lt/total_clean:.4f}")
        print(f"  Train ACC={acc:.4f}, Train ASR={asr:.4f} (r={r:.3f})")
        history["epoch"].append(epoch+1)
        history["loss_clean"].append(ep_lc/total_clean)
        history["loss_bd"].append(ep_lb/total_bd)
        history["loss_total"].append(ep_lt/total_clean)
        history["train_acc"].append(acc)
        history["train_asr"].append(asr)

    adv_vision.eval()
    classification_head.eval()
    return history

# Run Stage 2 with smaller batch to save memory
print("\nLoading models for FI training...")
adv_vision = load_vision_model_from_sd(finetuned_sds["CIFAR100"])
adv_vision.train()
adv_head = copy.deepcopy(classification_heads["CIFAR100"])

_m, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED)
ptm_frozen = _m.visual; del _m
ptm_frozen.eval()

# Use smaller batch DataLoader for FI training to save memory
fi_train_dl = DataLoader(
    train_loaders["CIFAR100"].dataset,
    batch_size=32,
    shuffle=True, num_workers=0, pin_memory=True
)

fi_history = train_badmerging_fi(
    adv_vision=adv_vision, ptm_vision=ptm_frozen,
    classification_head=adv_head, train_loader=fi_train_dl,
    optimized_trigger=optimized_trigger, config=ATTACK_CONFIG, device=str(device),
)

print("\nUpdating CIFAR100 backdoor model (FI enhanced)...")
adv_vision.eval().cpu()
enhanced_sd = adv_vision.state_dict()
finetuned_sds_original_cifar = copy.deepcopy(finetuned_sds["CIFAR100"])
finetuned_sds["CIFAR100"] = enhanced_sd
classification_heads["CIFAR100"] = adv_head.cpu()
print(f"  enhanced_sd keys: {len(enhanced_sd)}")
print("  finetuned_sds['CIFAR100'] updated to FI enhanced version!")

torch.save({"model_state_dict": enhanced_sd, "fi_history": fi_history},
           "./badmerging_convnext_enhanced_model.pth")
print("Stage 2 done!")

del ptm_frozen, adv_vision, finetuned_sds_original_cifar
torch.cuda.empty_cache()

# Plot
import numpy as np

def _smooth(arr, k=50):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < k:
        return arr
    return np.convolve(arr, np.ones(k)/k, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: per-batch loss components (smoothed)
sl_clean = _smooth(fi_history["step_loss_clean"])
sl_bd    = _smooth(fi_history["step_loss_bd"])
sl_total = _smooth(fi_history["step_loss_total"])
xs = np.arange(len(sl_clean))
axes[0].plot(xs, sl_clean, "b-",  label="L_clean", linewidth=1.5)
axes[0].plot(xs, sl_bd,    "r-",  label="L_bd",    linewidth=1.5)
axes[0].plot(xs, sl_total, "k--", label="L_total", linewidth=1.5)
axes[0].set_xlabel("Batch step")
axes[0].set_ylabel("Loss (50-batch moving avg)")
axes[0].set_title("FI Loss components per batch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: Train ACC per epoch (saturates fast, kept for completeness)
axes[1].plot(fi_history["epoch"], fi_history["train_acc"], "g-o", linewidth=2, markersize=8)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Train accuracy")
axes[1].set_title("Clean Accuracy (train)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3)

# Panel 3: Train ASR per epoch
axes[2].plot(fi_history["epoch"], fi_history["train_asr"], "r-o", linewidth=2, markersize=8)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Train ASR")
axes[2].set_title("Attack Success Rate (train)")
axes[2].set_ylim(0, 1.05)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fi_loss_training.png", dpi=150)
plt.show()


In [ ]:
print("Algorithm 1: Task Arithmetic")

SCALING_COEF = 0.3
flat_ptm = state_dict_to_vector(pretrained_sd)
print(f"pretrained vector dim: {flat_ptm.shape[0]:,}")

merged_flat = flat_ptm.clone()
print("\nStreaming task vector computation:")
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv = ft_vec - flat_ptm
    merged_flat = merged_flat + SCALING_COEF * tv
    print(f"  {task_name:10s} TV L2 norm: {tv.norm().item():.4f}  OK")
    del ft_vec, tv

ta_merged_sd = vector_to_state_dict(merged_flat, pretrained_sd)
del merged_flat; import gc; gc.collect()
print(f"\nTask Arithmetic merge done (lambda={SCALING_COEF})")
# No BN recalibration needed - ConvNeXt uses LayerNorm!
print(f"merged_sd keys: {len(ta_merged_sd)}")
evaluate_all_tasks(ta_merged_sd, "Task Arithmetic", all_results)


In [ ]:
print("Algorithm 2: TIES Merging")

K = 20; MERGE_FUNC = "dis-sum"; TIES_SCALING = 0.3
print(f"Params: K={K}%, merge_func={MERGE_FUNC}, lambda={TIES_SCALING}")

print("\nComputing task vector matrix...")
tv_flat_list = []
for task_name in task_names_ordered:
    ft_vec = state_dict_to_vector(finetuned_sds[task_name])
    tv_flat_list.append(ft_vec - flat_ptm)
    del ft_vec
    print(f"  {task_name:10s}: OK")

tv_flat = torch.vstack(tv_flat_list); del tv_flat_list
import gc; gc.collect()
print(f"TV matrix shape: {tv_flat.shape}")

merged_tv = ties_merging(tv_flat, reset_thresh=K, merge_func=MERGE_FUNC)
ties_merged_flat = flat_ptm + TIES_SCALING * merged_tv
del tv_flat, merged_tv; gc.collect(); torch.cuda.empty_cache()

ties_merged_sd = vector_to_state_dict(ties_merged_flat, pretrained_sd)
del ties_merged_flat; gc.collect()
print(f"\nTIES Merging done! merged_sd keys: {len(ties_merged_sd)}")
# No BN recalibration needed!
evaluate_all_tasks(ties_merged_sd, "TIES Merging", all_results)
del flat_ptm; gc.collect(); torch.cuda.empty_cache()


In [ ]:
print("Algorithm 3: RegMean (Gram-weighted)")

def regmean_merge(finetuned_sds_dict, pretrained_sd, task_names, test_loaders, device, a=0.1):
    """RegMean: Gram-weighted merge for Linear layers, simple avg for others.
    ConvNeXt has many Linear layers (in MLP blocks) so RegMean works well!"""
    all_params = {}
    all_grams = []

    for task_name in task_names:
        vision = load_vision_model_from_sd(finetuned_sds_dict[task_name])
        vision.eval().to(device)

        for name, param in vision.named_parameters():
            if name not in all_params: all_params[name] = []
            all_params[name].append(param.detach().cpu())

        grams, hooks, xn = {}, [], {}

        def make_hook(module_name):
            def hook_fn(module, input, output):
                x = input[0].detach()
                x = x.reshape(-1, x.size(-1))
                xtx = torch.matmul(x.t(), x)
                if module_name not in grams:
                    grams[module_name] = xtx / x.size(0)
                    xn[module_name] = x.size(0)
                else:
                    n_old = xn[module_name]
                    grams[module_name] = (grams[module_name] * n_old + xtx) / (x.size(0) + n_old)
                    xn[module_name] += x.size(0)
            return hook_fn

        for name, module in vision.named_modules():
            if isinstance(module, nn.Linear):
                hooks.append(module.register_forward_hook(make_hook(name)))

        loader = test_loaders.get(task_name)
        if loader:
            for bi, (imgs, _) in enumerate(loader):
                if bi >= 5: break
                with torch.no_grad(): vision(imgs.to(device))

        for h in hooks: h.remove()
        grams_cpu = {k: v.cpu() for k, v in grams.items()}
        all_grams.append(grams_cpu)
        del vision; torch.cuda.empty_cache()
        print(f"    {task_name}: {len(grams_cpu)} Gram matrices collected")

    merged_params = {}
    regmean_count = 0

    for name in all_params:
        h_avged = False
        if name.endswith('.weight'):
            mn = name[:-len('.weight')]
            if mn in all_grams[0]:
                regmean_count += 1
                gram_m_ws, gram_list = [], []
                for mid in range(len(task_names)):
                    if mn in all_grams[mid]:
                        pg = reduce_non_diag(all_grams[mid][mn], a=a)
                        gram_m_ws.append(torch.matmul(pg, all_params[name][mid].t()))
                        gram_list.append(pg)
                if gram_list:
                    try:
                        wt = torch.matmul(torch.linalg.pinv(sum(gram_list)), sum(gram_m_ws))
                        merged_params[name] = wt.t()
                        h_avged = True
                    except Exception as e:
                        print(f"    [WARN] RegMean failed for {name}: {e}")

        if not h_avged:
            merged_params[name] = torch.stack(all_params[name], 0).mean(0)

    print(f"  RegMean: {regmean_count} Linear layers Gram-weighted, rest simple-averaged")
    return merged_params

regmean_params = regmean_merge(finetuned_sds, pretrained_sd, task_names_ordered, test_loaders, device)
regmean_merged_sd = copy.deepcopy(pretrained_sd)
matched = 0
for k, v in regmean_params.items():
    if k in regmean_merged_sd: regmean_merged_sd[k] = v; matched += 1
print(f"RegMean merge done! matched={matched}")
# No BN recalibration needed!
print(f"merged_sd keys: {len(regmean_merged_sd)}")
evaluate_all_tasks(regmean_merged_sd, "RegMean", all_results)


In [ ]:
print("Algorithm 4: Simple Averaging")

simple_avg_sd = {}
for key in pretrained_sd:
    params_list = [finetuned_sds[tn][key].float() for tn in task_names_ordered if key in finetuned_sds[tn]]
    simple_avg_sd[key] = torch.stack(params_list, 0).mean(0) if params_list else pretrained_sd[key].clone()
print(f"Simple Averaging done! merged_sd keys: {len(simple_avg_sd)}")
# No BN recalibration needed!
evaluate_all_tasks(simple_avg_sd, "Simple Averaging", all_results)


In [ ]:
import pandas as pd

methods = list(all_results.keys())
rows = []
for method in methods:
    res = all_results[method]
    row = {"Method": method}
    for tn in task_names_ordered:
        if tn in res["cda"]: row[f"{tn}_CDA"] = res["cda"][tn] * 100
    row["Avg_CDA"] = res["avg_cda"] * 100
    row["ASR"] = res["asr"] * 100 if res["asr"] else None
    rows.append(row)

df_results = pd.DataFrame(rows)
print("\nResults table:")
print(df_results.to_string(index=False, float_format="%.2f"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
bars1 = axes[0].bar(methods, df_results["Avg_CDA"], color=colors[:len(methods)], edgecolor='black')
axes[0].set_ylabel("Average CDA (%)"); axes[0].set_title("Average CDA (ConvNeXt)"); axes[0].set_ylim(0, 100)
for b, v in zip(bars1, df_results["Avg_CDA"]): axes[0].text(b.get_x()+b.get_width()/2., b.get_height()+1, f'{v:.1f}%', ha='center')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(axis='y', alpha=0.3)

asr_vals = df_results["ASR"].fillna(0)
colors2 = ['#FF6B6B', '#EE5A24', '#F8B739', '#FDA7DF']
bars2 = axes[1].bar(methods, asr_vals, color=colors2[:len(methods)], edgecolor='black')
axes[1].set_ylabel("ASR (%)"); axes[1].set_title("ASR (ConvNeXt)"); axes[1].set_ylim(0, 105)
axes[1].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90%')
for b, v in zip(bars2, asr_vals): axes[1].text(b.get_x()+b.get_width()/2., b.get_height()+1, f'{v:.1f}%', ha='center')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y', alpha=0.3); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
print("""
Threat Model:
  - Attacker controls ONE task model (CIFAR-100)
  - Attacker does NOT know which merging algorithm will be used
  - Stage 2 FI Loss optimizes over random feature interpolations
    r ~ U[0.1, 1.0], making it algorithm-agnostic
  - The SAME backdoored model is evaluated against ALL 4 algorithms
  - Architecture: CLIP ConvNeXt (Modern CNN with LayerNorm, NOT Transformer)
""")

algorithms = ["Task Arithmetic", "TIES Merging", "RegMean", "Simple Averaging"]
cross_data = []
for algo in algorithms:
    if algo in all_results:
        res = all_results[algo]
        cross_data.append({
            "Merging Algorithm": algo,
            "ASR (%)": res["asr"]*100 if res["asr"] else 0,
            "Avg CDA (%)": res["avg_cda"]*100,
            **{f"{tn} CDA (%)": res["cda"].get(tn, 0)*100 for tn in task_names_ordered}
        })
df_cross = pd.DataFrame(cross_data)
print("Cross-Algorithm Results (Single Backdoor Model, ConvNeXt):")
print(df_cross.to_string(index=False, float_format="%.2f"))

asr_list = [d["ASR (%)"] for d in cross_data]
print(f"""
{'='*50}
Black-Box Robustness Summary (ConvNeXt):
  ASR across 4 unknown algorithms:
    Mean: {np.mean(asr_list):.2f}% | Std: {np.std(asr_list):.2f}%
    Min:  {min(asr_list):.2f}% ({cross_data[np.argmin(asr_list)]['Merging Algorithm']})
    Max:  {max(asr_list):.2f}% ({cross_data[np.argmax(asr_list)]['Merging Algorithm']})
  CDA range: {min(d['Avg CDA (%)'] for d in cross_data):.2f}% - {max(d['Avg CDA (%)'] for d in cross_data):.2f}%

Conclusion: FI Loss produces algorithm-agnostic backdoors even on CNN (ConvNeXt).
{'='*50}
""")

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(algorithms))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
bars = ax.bar(x, asr_list, color=colors, edgecolor='black')
ax.set_ylabel("ASR (%)"); ax.set_title("Black-Box ASR: Same Backdoor vs Unknown Algorithm (ConvNeXt)")
ax.set_ylim(0, 105); ax.set_xticks(x); ax.set_xticklabels(algorithms, rotation=15)
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
for b, v in zip(bars, asr_list): ax.text(b.get_x()+b.get_width()/2., b.get_height()+1, f'{v:.1f}%', ha='center', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:

import gc

# Re-define normalize_vision_sd in case of kernel restart
def normalize_vision_sd(raw_sd, reference_keys):
    sd = {}
    for k, v in raw_sd.items():
        if k in reference_keys:
            sd[k] = v
        else:
            for prefix in ["visual.", "module.", "model.visual."]:
                stripped = k[len(prefix):] if k.startswith(prefix) else None
                if stripped and stripped in reference_keys:
                    sd[stripped] = v
                    break
    return sd

print("\nUsing clean CIFAR-100 model from memory backup...")
clean_cifar_sd = normalize_vision_sd(clean_cifar100_sd_backup, PTM_KEYS)
for mk in PTM_KEYS - set(clean_cifar_sd.keys()):
    clean_cifar_sd[mk] = pretrained_sd[mk].clone()
for ek in set(clean_cifar_sd.keys()) - PTM_KEYS:
    del clean_cifar_sd[ek]
assert set(clean_cifar_sd.keys()) == PTM_KEYS
print(f"  clean CIFAR-100: {len(clean_cifar_sd)} keys")

print("Training clean CIFAR-100 head on clean fine-tuned encoder features...")
clean_cifar_head = nn.Linear(FEATURE_DIM, 100)
nn.init.xavier_uniform_(clean_cifar_head.weight)
nn.init.zeros_(clean_cifar_head.bias)
_clean_cifar_vision = load_vision_model_from_sd(clean_cifar_sd)
clean_train_dl = train_loaders.get("CIFAR100", test_loaders["CIFAR100"])
clean_cifar_head = train_classification_head(
    _clean_cifar_vision, clean_cifar_head, clean_train_dl, device, epochs=5, lr=1e-3)
clean_cifar_head = clean_cifar_head.cpu()
del _clean_cifar_vision; torch.cuda.empty_cache()

clean_finetuned_sds = {}
for tn in task_names_ordered:
    clean_finetuned_sds[tn] = clean_cifar_sd if TASK_CONFIG[tn]["is_adversary"] else finetuned_sds[tn]

baseline_heads = dict(classification_heads)
baseline_heads["CIFAR100"] = clean_cifar_head
baseline_results = {}

def evaluate_clean_baseline(merged_sd, method_name):
    vision = load_vision_model_from_sd(merged_sd)
    vision.eval().to(device)
    results = {}
    for tn in task_names_ordered:
        if tn not in test_loaders: continue
        head = baseline_heads[tn].to(device)
        cda = evaluate_clean_accuracy(vision, head, test_loaders[tn], device)
        results[tn] = cda; head.cpu()
    del vision; torch.cuda.empty_cache()
    avg_cda = np.mean(list(results.values()))
    results["avg"] = avg_cda
    print(f"\n--- Clean {method_name} ---")
    for tn in task_names_ordered:
        if tn in results: print(f"  {tn:10s}: CDA={results[tn]*100:.2f}%")
    print(f"  {'Avg CDA':10s}: {avg_cda*100:.2f}%")
    baseline_results[method_name] = results
    return results

# 1. Task Arithmetic
print("\n[1/4] Clean Task Arithmetic...")
clean_flat_ptm = state_dict_to_vector(pretrained_sd)
ta_bl = clean_flat_ptm.clone()
for tn in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[tn])
    ta_bl = ta_bl + 0.3 * (ft_vec - clean_flat_ptm); del ft_vec
clean_ta_results = evaluate_clean_baseline(
    vector_to_state_dict(ta_bl, pretrained_sd), "Task Arithmetic")
del ta_bl; gc.collect(); torch.cuda.empty_cache()

# 2. TIES
print("\n[2/4] Clean TIES Merging...")
tv_list = []
for tn in task_names_ordered:
    ft_vec = state_dict_to_vector(clean_finetuned_sds[tn])
    tv_list.append(ft_vec - clean_flat_ptm); del ft_vec
tv_matrix = torch.vstack(tv_list); del tv_list; gc.collect()

sample_size = int(tv_matrix.shape[1] * 0.1)
indices = torch.randperm(tv_matrix.shape[1])[:sample_size]
threshold = torch.quantile(tv_matrix[:, indices].abs(), 20/100.0, dim=1, keepdim=True)
tv_trimmed = tv_matrix.clone()
tv_trimmed[tv_matrix.abs() < threshold] = 0.0
del threshold
sign_votes = tv_trimmed.sign().sum(dim=0)
majority_sign = sign_votes.sign()
majority_sign[majority_sign == 0] = 1
aligned = tv_trimmed.clone()
for i in range(aligned.shape[0]):
    mask = aligned[i].sign() != majority_sign
    aligned[i][mask] = 0.0
clean_ties_flat = clean_flat_ptm + 0.3 * aligned.sum(dim=0)
del tv_matrix, tv_trimmed, aligned; gc.collect()
clean_ties_results = evaluate_clean_baseline(
    vector_to_state_dict(clean_ties_flat, pretrained_sd), "TIES Merging")
del clean_ties_flat; gc.collect(); torch.cuda.empty_cache()

# 3. RegMean (Gram-weighted)
print("\n[3/4] Clean RegMean (Gram-weighted)...")
cl_all_params, cl_all_grams = {}, []
for tn in task_names_ordered:
    vis = load_vision_model_from_sd(clean_finetuned_sds[tn]); vis.eval().to(device)
    for n, p in vis.named_parameters():
        if n not in cl_all_params: cl_all_params[n] = []
        cl_all_params[n].append(p.detach().cpu())
    grams, hooks, xn = {}, [], {}
    def make_hook(mn):
        def hf(m, inp, out):
            x = inp[0].detach().reshape(-1, inp[0].size(-1))
            xtx = torch.matmul(x.t(), x)
            if mn not in grams: grams[mn] = xtx/x.size(0); xn[mn] = x.size(0)
            else: grams[mn] = (grams[mn]*xn[mn]+xtx)/(x.size(0)+xn[mn]); xn[mn] += x.size(0)
        return hf
    for n, m in vis.named_modules():
        if isinstance(m, nn.Linear): hooks.append(m.register_forward_hook(make_hook(n)))
    loader = test_loaders.get(tn)
    if loader:
        for bi, (imgs, _) in enumerate(loader):
            if bi >= 5: break
            with torch.no_grad(): vis(imgs.to(device))
    for h in hooks: h.remove()
    cl_all_grams.append({k: v.cpu() for k, v in grams.items()})
    del vis; torch.cuda.empty_cache()
    print(f"  {tn}: {len(cl_all_grams[-1])} Gram matrices")

cl_rm_merged = {}; rc = 0
for name in cl_all_params:
    done = False
    if name.endswith('.weight'):
        mn = name[:-len('.weight')]
        if len(cl_all_grams) > 0 and mn in cl_all_grams[0]:
            rc += 1; gws, gl = [], []
            for mid in range(len(task_names_ordered)):
                if mn in cl_all_grams[mid]:
                    pg = reduce_non_diag(cl_all_grams[mid][mn], a=0.1)
                    gws.append(torch.matmul(pg, cl_all_params[name][mid].t())); gl.append(pg)
            if gl:
                try:
                    cl_rm_merged[name] = torch.matmul(torch.linalg.pinv(sum(gl)), sum(gws)).t(); done = True
                except: pass
    if not done: cl_rm_merged[name] = torch.stack(cl_all_params[name], 0).mean(0)
print(f"  RegMean merged {rc} Linear layers")
cl_rm_sd = copy.deepcopy(pretrained_sd)
for k, v in cl_rm_merged.items():
    if k in cl_rm_sd: cl_rm_sd[k] = v
del cl_rm_merged, cl_all_params, cl_all_grams
clean_regmean_results = evaluate_clean_baseline(cl_rm_sd, "RegMean")
del cl_rm_sd; gc.collect(); torch.cuda.empty_cache()

# 4. Simple Averaging
print("\n[4/4] Clean Simple Averaging...")
cl_sa_sd = {}
for key in pretrained_sd:
    ps = [clean_finetuned_sds[tn][key].float() for tn in task_names_ordered if key in clean_finetuned_sds[tn]]
    cl_sa_sd[key] = torch.stack(ps).mean(0) if ps else pretrained_sd[key].clone()
clean_sa_results = evaluate_clean_baseline(cl_sa_sd, "Simple Averaging")
del cl_sa_sd

# Delta CDA table
print("\n" + "=" * 80)
attack_methods = ["Task Arithmetic", "TIES Merging", "RegMean", "Simple Averaging"]
clean_all = {"Task Arithmetic": clean_ta_results, "TIES Merging": clean_ties_results,
             "RegMean": clean_regmean_results, "Simple Averaging": clean_sa_results}

print(f"\n{'Method':20s} | {'Baseline CDA':>12s} | {'Attack CDA':>10s} | {'CDA Drop':>8s} | {'ASR':>6s}")
for method in attack_methods:
    if method in all_results and method in clean_all:
        b_cda = clean_all[method]["avg"] * 100
        a_cda = all_results[method]["avg_cda"] * 100
        a_asr = all_results[method].get("asr", 0)
        a_asr = a_asr * 100 if a_asr else 0
        drop = b_cda - a_cda
        print(f"{method:20s} | {b_cda:11.2f}% | {a_cda:9.2f}% | {drop:+7.2f}% | {a_asr:5.1f}%")

del clean_flat_ptm, clean_finetuned_sds
gc.collect(); torch.cuda.empty_cache()
print("\nClean baseline done!")


In [ ]:
from sklearn.manifold import TSNE

# Define seed for reproducibility
SEED = 42

print("  t-SNE Feature Visualization")

tsne_vision = load_vision_model_from_sd(regmean_merged_sd)
tsne_vision.eval().to(device)

N_SAMPLES = 500
all_indices = list(range(len(cifar100_dataset)))
random.shuffle(all_indices)
non_target_idx = [i for i in all_indices if cifar100_test.targets[i] != TARGET_CLASS][:N_SAMPLES]
clean_idx = all_indices[:N_SAMPLES]

tsne_trigger = optimized_trigger.cpu()

@torch.no_grad()
def extract_features_tsne(indices, add_trig=False):
    feats, labs = [], []
    for idx in tqdm(indices, desc="Extracting"):
        img, label = cifar100_dataset[idx]
        if add_trig: img = add_trigger(img, tsne_trigger)
        feat = tsne_vision(img.unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        feats.append(feat); labs.append(label)
    return np.array(feats), np.array(labs)

print("Extracting clean features...")
clean_feats, clean_labs = extract_features_tsne(clean_idx, False)
print("Extracting triggered features...")
trig_feats, trig_labs = extract_features_tsne(non_target_idx, True)
del tsne_vision; torch.cuda.empty_cache()

print(f"\nRunning t-SNE (perplexity=30, n_iter=1000)...")
all_feats = np.vstack([clean_feats, trig_feats])
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, learning_rate='auto')
emb_2d = tsne.fit_transform(all_feats)
clean_emb, trig_emb = emb_2d[:len(clean_feats)], emb_2d[len(clean_feats):]
print(f"t-SNE done: {emb_2d.shape}")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
target_mask = clean_labs == TARGET_CLASS
ax = axes[0]; ax.scatter(clean_emb[~target_mask,0], clean_emb[~target_mask,1], c='#3498db', s=10, alpha=0.3, label='Other')
if target_mask.sum()>0: ax.scatter(clean_emb[target_mask,0], clean_emb[target_mask,1], c='gold', s=80, marker='*', edgecolors='black', label=f'Target', zorder=10)
ax.set_title("Clean Images"); ax.legend()

ax = axes[1]; ax.scatter(trig_emb[:,0], trig_emb[:,1], c='#e74c3c', s=15, alpha=0.5, marker='x', label='Triggered')
if target_mask.sum()>0: ax.scatter(clean_emb[target_mask,0], clean_emb[target_mask,1], c='gold', s=80, marker='*', edgecolors='black', label='Target (clean)', zorder=10)
ax.set_title("Triggered Images"); ax.legend()

ax = axes[2]; ax.scatter(clean_emb[~target_mask,0], clean_emb[~target_mask,1], c='#3498db', s=10, alpha=0.2, label='Clean other')
ax.scatter(trig_emb[:,0], trig_emb[:,1], c='#e74c3c', s=15, alpha=0.3, marker='x', label='Triggered')
if target_mask.sum()>0: ax.scatter(clean_emb[target_mask,0], clean_emb[target_mask,1], c='gold', s=80, marker='*', edgecolors='black', label='Target', zorder=10)
ax.set_title("Overlay"); ax.legend()
plt.suptitle("t-SNE: BadMerging on CLIP ConvNeXt (CNN)", fontsize=14)
plt.tight_layout(); plt.show()

# Distance analysis
if target_mask.sum() > 0:
    target_centroid = clean_emb[target_mask].mean(axis=0)
    clean_dists = np.linalg.norm(clean_emb[~target_mask] - target_centroid, axis=1)
    trig_dists = np.linalg.norm(trig_emb - target_centroid, axis=1)
    print(f"\nClean -> target centroid: median={np.median(clean_dists):.1f}")
    print(f"Triggered -> target centroid: median={np.median(trig_dists):.1f}")
    print(f"Distance ratio: {np.median(clean_dists)/np.median(trig_dists):.2f}x")
print("\nt-SNE visualization done!")

In [ ]:
SAVE_DIR = "./merging_output_convnext"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving results...")
results_json = {}
for method, res in all_results.items():
    results_json[method] = {
        "avg_cda": round(res["avg_cda"]*100, 2),
        "asr": round(res["asr"]*100, 2) if res["asr"] else None,
        "per_task_cda": {k: round(v*100, 2) for k, v in res["cda"].items()},
    }
results_path = os.path.join(SAVE_DIR, "merging_results_convnext.json")
with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2)
print(f"Results JSON: {results_path}")
print(json.dumps(results_json, indent=2))

# Save trigger
torch.save(optimized_trigger, os.path.join(SAVE_DIR, "optimized_trigger_convnext.pth"))
print(f"\nAll results saved to {SAVE_DIR}/")
